<a href="https://colab.research.google.com/github/pvallejoieu2024-spec/pvallejoieu2024-spec.github.io/blob/main/AI_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PREPARE THE DATA

In [ ]:
!pip install -q pymupdf pandas

#1. Download the data

In [43]:
from google.colab import drive
from pathlib import Path

# Connect Google Drive
drive.mount('/content/drive')

# Folder containing the medication PDFs
DATA_DIR = Path("/content/drive/MyDrive/AI MACHINE PROJECT/data")

# Find all PDF files
pdf_files = sorted(DATA_DIR.glob("*.pdf"))

print("Total PDFs found:", len(pdf_files))
print("\nPDF filenames:")

for pdf in pdf_files:
    print("-", pdf.name)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total PDFs found: 30

PDF filenames:
- abasaglar.pdf
- actos.pdf
- aprovel.pdf
- brilique.pdf
- corlentor.pdf
- duoplavin.pdf
- efient.pdf
- eliquis.pdf
- entresto.pdf
- eucreas.pdf
- forxiga.pdf
- insuman.pdf
- jardiance.pdf
- kerendia.pdf
- leqvio.pdf
- lixiana.pdf
- micardis.pdf
- mounjaro.pdf
- multaq.pdf
- nilemdo.pdf
- ozempic.pdf
- plavix.pdf
- pradaxa.pdf
- praluent.pdf
- ranexa.pdf
- repatha.pdf
- synjardy.pdf
- verquvo.pdf
- xarelto.pdf
- xelevia.pdf


#2. Identified how many SmPCs each PDF contains

In [ ]:
import fitz
import re
import pandas as pd

results = []

for pdf_path in pdf_files:
    doc = fitz.open(pdf_path)
    count = 0

    for page in doc:
        text = page.get_text("text")

        # Stop when Annex II begins
        if re.search(r"(?im)^\s*ANNEX\s+II\s*$", text):
            break

        # Count SmPC starts
        count += len(re.findall(
            r"(?im)^\s*1\.\s*NAME OF THE MEDICINAL PRODUCT\s*$",
            text
        ))

    results.append({
        "Medicine": pdf_path.stem.capitalize(),
        "Number of SmPCs": count,
        "Classification": "Multiple SmPCs" if count > 1 else "Single SmPC"
    })

    doc.close()

df_smpcs = pd.DataFrame(results)

single = df_smpcs[df_smpcs["Number of SmPCs"] == 1]
multiple = df_smpcs[df_smpcs["Number of SmPCs"] > 1]

print("========== MEDICINES WITH 1 SmPC ==========")
display(single)

print("\n========== MEDICINES WITH MULTIPLE SmPCs ==========")
display(multiple)

print("\nSUMMARY")
print("Medicines with 1 SmPC:", len(single))
print("Medicines with multiple SmPCs:", len(multiple))
print("Total SmPCs:", df_smpcs["Number of SmPCs"].sum())

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


========== MEDICINES WITH 1 SmPC ==========


,Medicine,Number of SmPCs,Classification
1,Actos,1,Single SmPC
4,Corlentor,1,Single SmPC
5,Duoplavin,1,Single SmPC
6,Efient,1,Single SmPC
9,Eucreas,1,Single SmPC
10,Forxiga,1,Single SmPC
12,Jardiance,1,Single SmPC
13,Kerendia,1,Single SmPC
14,Leqvio,1,Single SmPC
15,Lixiana,1,Single SmPC



========== MEDICINES WITH MULTIPLE SmPCs ==========


,Medicine,Number of SmPCs,Classification
0,Abasaglar,2,Multiple SmPCs
2,Aprovel,6,Multiple SmPCs
3,Brilique,3,Multiple SmPCs
7,Eliquis,4,Multiple SmPCs
8,Entresto,2,Multiple SmPCs
11,Insuman,9,Multiple SmPCs
22,Pradaxa,4,Multiple SmPCs
24,Ranexa,3,Multiple SmPCs
28,Xarelto,6,Multiple SmPCs



SUMMARY
Medicines with 1 SmPC: 21
Medicines with multiple SmPCs: 9
Total SmPCs: 60


In [ ]:
import fitz
import re
import pandas as pd

START_RE = re.compile(
    r"(?im)^[ \t]*1\s*\.\s*NAME OF THE MEDICINAL PRODUCT[ \t]*$"
)

ANNEX_RE = re.compile(
    r"(?im)^[ \t]*ANNEX\s+II[ \t]*$"
)

all_smpcs = []
report = []

for pdf_path in pdf_files:
    doc = fitz.open(pdf_path)

    # Find actual SmPC starts before Annex II
    starts = []

    for page_num, page in enumerate(doc):
        text = page.get_text("text")

        # Stop at Annex II
        if ANNEX_RE.search(text):
            break

        for match in START_RE.finditer(text):
            starts.append((page_num, match.start()))

    # Extract text between consecutive SmPC starts
    for i, (start_page, start_pos) in enumerate(starts):

        if i + 1 < len(starts):
            end_page, end_pos = starts[i + 1]
        else:
            end_page, end_pos = page_num, None

        parts = []

        for p in range(start_page, end_page + 1):
            page_text = doc[p].get_text("text")

            if p == start_page:
                page_text = page_text[start_pos:]

            if p == end_page and end_pos is not None:
                if p == start_page:
                    page_text = doc[p].get_text("text")[start_pos:end_pos]
                else:
                    page_text = page_text[:end_pos]

            parts.append(page_text)

        all_smpcs.append({
            "drug": pdf_path.stem.capitalize(),
            "smpc_id": f"{pdf_path.stem.lower()}_{i+1:02d}",
            "source_pdf": pdf_path.name,
            "text": "\n".join(parts)
        })

    report.append({
        "drug": pdf_path.stem.capitalize(),
        "smpc_count": len(starts)
    })

    doc.close()

smpcs_df = pd.DataFrame(all_smpcs)
report_df = pd.DataFrame(report)

display(report_df)

print("\nTotal medicines:", len(report_df))
print("Total SmPCs:", len(smpcs_df))

,drug,smpc_count
0,Abasaglar,2
1,Actos,1
2,Aprovel,6
3,Brilique,3
4,Corlentor,1
5,Duoplavin,1
6,Efient,1
7,Eliquis,4
8,Entresto,2
9,Eucreas,1



Total medicines: 30
Total SmPCs: 60


#3. Remove Annex II from the extracted SmPCs

In [ ]:
import re
import pandas as pd

validation = []

for _, row in smpcs_df.iterrows():
    text = row["text"]

    validation.append({
        "drug": row["drug"],
        "smpc_id": row["smpc_id"],
        "characters": len(text),
        "has_section_1": bool(re.search(
            r"(?im)^1\s*\.\s*NAME OF THE MEDICINAL PRODUCT",
            text
        )),
        "has_section_10": bool(re.search(
            r"(?im)^10\s*\.\s*DATE OF REVISION OF THE TEXT",
            text
        )),
        "contains_annex_ii": bool(re.search(
            r"(?im)^\s*ANNEX\s+II\s*$",
            text
        ))
    })

validation_df = pd.DataFrame(validation)

print("Total SmPCs:", len(validation_df))
print("Missing Section 1:", (~validation_df["has_section_1"]).sum())
print("Missing Section 10:", (~validation_df["has_section_10"]).sum())
print("Containing Annex II:", validation_df["contains_annex_ii"].sum())

print("\nSmPCs WITH POTENTIAL PROBLEMS:")
display(validation_df[
    (~validation_df["has_section_1"]) |
    (~validation_df["has_section_10"]) |
    (validation_df["contains_annex_ii"])
])

Total SmPCs: 60
Missing Section 1: 0
Missing Section 10: 0
Containing Annex II: 30

SmPCs WITH POTENTIAL PROBLEMS:


,drug,smpc_id,characters,has_section_1,has_section_10,contains_annex_ii
1,Abasaglar,abasaglar_02,35948,True,True,True
2,Actos,actos_01,36932,True,True,True
8,Aprovel,aprovel_06,39179,True,True,True
11,Brilique,brilique_03,65661,True,True,True
12,Corlentor,corlentor_01,46884,True,True,True
13,Duoplavin,duoplavin_01,73669,True,True,True
14,Efient,efient_01,47641,True,True,True
18,Eliquis,eliquis_04,67825,True,True,True
20,Entresto,entresto_02,58525,True,True,True
21,Eucreas,eucreas_01,49615,True,True,True


In [ ]:
import re

ANNEX_RE = re.compile(
    r"(?im)^[ \t]*ANNEX[ \t]+II[ \t]*$"
)

# Work on a copy
smpcs_clean = smpcs_df.copy()

removed = []

for idx, row in smpcs_clean.iterrows():
    text = row["text"]

    # Find Annex II after Section 10
    section_10 = re.search(
        r"(?im)^[ \t]*10\s*\.\s*DATE OF REVISION OF THE TEXT",
        text
    )

    if section_10:
        annex_matches = [
            m for m in ANNEX_RE.finditer(text)
            if m.start() > section_10.end()
        ]

        if annex_matches:
            cut_position = annex_matches[0].start()

            removed.append({
                "drug": row["drug"],
                "smpc_id": row["smpc_id"],
                "characters_removed": len(text) - cut_position
            })

            smpcs_clean.at[idx, "text"] = text[:cut_position].strip()

removed_df = pd.DataFrame(removed)

print("Total SmPCs:", len(smpcs_clean))
print("SmPCs trimmed:", len(removed_df))

display(removed_df)

Total SmPCs: 60
SmPCs trimmed: 30


,drug,smpc_id,characters_removed
0,Abasaglar,abasaglar_02,356
1,Actos,actos_01,296
2,Aprovel,aprovel_06,295
3,Brilique,brilique_03,320
4,Corlentor,corlentor_01,304
5,Duoplavin,duoplavin_01,301
6,Efient,efient_01,299
7,Eliquis,eliquis_04,297
8,Entresto,entresto_02,296
9,Eucreas,eucreas_01,298


In [ ]:
remaining = []

for _, row in smpcs_clean.iterrows():
    if ANNEX_RE.search(row["text"]):
        remaining.append(row["smpc_id"])

print("Total SmPCs:", len(smpcs_clean))
print("SmPCs still containing Annex II:", len(remaining))
print("IDs with Annex II:", remaining)

Total SmPCs: 60
SmPCs still containing Annex II: 0
IDs with Annex II: []


#4. Extract individual sections from each SmPC

In [ ]:
# Inspect the raw text of the first extracted SmPC

text = smpcs_clean.iloc[0]["text"]

print("SmPC:", smpcs_clean.iloc[0]["smpc_id"])
print("Total characters:", len(text))

print("\n========== FIRST 100 LINES ==========\n")

for i, line in enumerate(text.splitlines()[:100], start=1):
    print(f"{i:03d}: {repr(line)}")

SmPC: abasaglar_01
Total characters: 35398

========== FIRST 100 LINES ==========

001: '1. '
002: 'NAME OF THE MEDICINAL PRODUCT '
003: ' '
004: 'ABASAGLAR 100 units/mL solution for injection in a cartridge '
005: ' '
006: ' '
007: '2. '
008: 'QUALITATIVE AND QUANTITATIVE COMPOSITION '
009: ' '
010: 'Each mL contains 100 units insulin glargine* (equivalent to 3.64 mg).  '
011: ' '
012: 'Each cartridge contains 3 mL of solution for injection, equivalent to 300 units.  '
013: ' '
014: '*produced by recombinant DNA technology in Escherichia coli.  '
015: ' '
016: 'For the full list of excipients, see section 6.1. '
017: ' '
018: ' '
019: '3. '
020: 'PHARMACEUTICAL FORM '
021: ' '
022: 'Solution for injection (injection). '
023: ' '
024: 'Clear, colourless solution. '
025: ' '
026: ' '
027: '4. '
028: 'CLINICAL PARTICULARS '
029: ' '
030: '4.1 '
031: 'Therapeutic indications  '
032: ' '
033: 'Treatment of diabetes mellitus in adults, adolescents and children aged 2 years and above. '
034:

In [ ]:
import re
import pandas as pd
from difflib import SequenceMatcher

SECTION_ORDER = list(SECTION_TITLES.keys())


def normalize_title(text):
    text = text.lower()
    text = re.sub(r"[-–—]", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return " ".join(text.split())


def title_matches(actual, expected):
    actual = normalize_title(actual)
    expected = normalize_title(expected)

    if actual == expected:
        return True

    if expected in actual or actual in expected:
        return len(actual) >= 8

    similarity = SequenceMatcher(
        None, actual, expected
    ).ratio()

    return similarity >= 0.82


def extract_sections(text):

    # Section number and title appear on separate lines.
    # Examples:
    # 4.1
    # Therapeutic indications
    #
    # 9.
    # DATE OF FIRST AUTHORISATION / RENEWAL...

    heading_re = re.compile(
        r"(?m)^[ \t]*(10|[1-9](?:\.[1-9])?)\.?[ \t]*\r?\n"
    )

    candidates = []

    for match in heading_re.finditer(text):

        number = match.group(1)

        if number not in SECTION_TITLES:
            continue

        # Inspect the following lines, including wrapped titles
        lines = text[match.end():].splitlines()[:5]
        lines = [line.strip() for line in lines if line.strip()]

        expected = SECTION_TITLES[number]

        matched_end = None

        for n in range(1, min(3, len(lines)) + 1):

            possible_title = " ".join(lines[:n])

            if title_matches(possible_title, expected):
                matched_end = match.end()
                break

        if matched_end is not None:
            candidates.append({
                "number": number,
                "start": match.start(),
                "end": match.end()
            })

    # Select headings in the expected SmPC order.
    # This avoids treating references or page numbers
    # inside the text as new sections.

    ordered = []
    previous_position = -1

    for number in SECTION_ORDER:

        possible = [
            candidate
            for candidate in candidates
            if candidate["number"] == number
            and candidate["start"] > previous_position
        ]

        if possible:
            selected = possible[0]
            ordered.append(selected)
            previous_position = selected["start"]

    # Restore document order
    ordered.sort(key=lambda x: x["start"])

    sections = []

    for i, current in enumerate(ordered):

        next_start = (
            ordered[i + 1]["start"]
            if i + 1 < len(ordered)
            else len(text)
        )

        number = current["number"]

        content = text[
            current["end"]:next_start
        ].strip()

        sections.append({
            "section_number": number,
            "section_title": SECTION_TITLES[number],
            "text": content,
            "chars": len(content)
        })

    return sections


# Extract all sections from the 60 SmPCs

all_sections = []
report = []

for _, smpc in smpcs_clean.iterrows():

    sections = extract_sections(smpc["text"])

    for section in sections:
        all_sections.append({
            "drug": smpc["drug"],
            "smpc_id": smpc["smpc_id"],
            "source_pdf": smpc["source_pdf"],
            **section
        })

    report.append({
        "drug": smpc["drug"],
        "smpc_id": smpc["smpc_id"],
        "sections_found": len(sections)
    })


sections_df = pd.DataFrame(
    all_sections,
    columns=[
        "drug",
        "smpc_id",
        "source_pdf",
        "section_number",
        "section_title",
        "text",
        "chars"
    ]
)

report_df = pd.DataFrame(report)


# Validation

print("SmPCs processed:", len(report_df))
print("Sections extracted:", len(sections_df))
print("Expected sections:", len(smpcs_clean) * len(SECTION_ORDER))

print("\nSections per SmPC:")
print(report_df["sections_found"].value_counts().sort_index())

print("\nSmPCs with fewer than 28 sections:")
display(report_df[report_df["sections_found"] < 28])

print("\nLargest extracted sections:")
display(
    sections_df.nlargest(10, "chars")[
        ["drug", "smpc_id", "section_number", "chars"]
    ]
)

SmPCs processed: 60
Sections extracted: 1680
Expected sections: 1680

Sections per SmPC:
sections_found
28    60
Name: count, dtype: int64

SmPCs with fewer than 28 sections:


,drug,smpc_id,sections_found



Largest extracted sections:


,drug,smpc_id,section_number,chars
1050,Mounjaro,mounjaro_01,5.1,58769
630,Forxiga,forxiga_01,5.1,49722
910,Jardiance,jardiance_01,5.1,44669
1414,Repatha,repatha_01,5.1,39578
434,Eliquis,eliquis_01,5.1,39018
1218,Pradaxa,pradaxa_02,5.1,36868
1302,Praluent,praluent_01,5.1,35072
1554,Xarelto,xarelto_03,5.1,34221
1582,Xarelto,xarelto_04,5.1,34205
462,Eliquis,eliquis_02,5.1,34074


In [ ]:
# Validate extracted sections

validation = []

for smpc_id, group in sections_df.groupby("smpc_id", sort=False):

    numbers = group["section_number"].tolist()

    correct_order = numbers == SECTION_ORDER

    empty_sections = group[
        group["text"].str.strip().eq("")
    ]["section_number"].tolist()

    validation.append({
        "smpc_id": smpc_id,
        "correct_order": correct_order,
        "empty_sections": empty_sections,
        "total_chars": group["chars"].sum()
    })

validation_df = pd.DataFrame(validation)

print("SmPCs with correct section order:",
      validation_df["correct_order"].sum(), "/60")

print("SmPCs with empty sections:",
      validation_df["empty_sections"].apply(bool).sum())

print("\nSmPCs with validation problems:")
display(
    validation_df[
        (~validation_df["correct_order"]) |
        (validation_df["empty_sections"].apply(bool))
    ]
)

SmPCs with correct section order: 60 /60
SmPCs with empty sections: 0

SmPCs with validation problems:


,smpc_id,correct_order,empty_sections,total_chars


In [ ]:
# Check text preservation across all 60 SmPCs

preservation_report = []

for _, smpc in smpcs_clean.iterrows():

    smpc_id = smpc["smpc_id"]
    original_text = smpc["text"]

    extracted = extract_sections(original_text)

    # Locate each extracted section's content in the original
    cursor = 0
    unmatched_parts = []

    for section in extracted:

        content = section["text"]

        position = original_text.find(content, cursor)

        if position == -1:
            unmatched_parts.append("CONTENT NOT FOUND")
            continue

        gap = original_text[cursor:position]

        # Remove expected section heading from the gap
        heading_pattern = (
            rf"(?s)^\s*{re.escape(section['section_number'])}"
            rf"\.?\s*"
        )

        gap = re.sub(heading_pattern, "", gap, count=1)

        if gap.strip():
            unmatched_parts.append(gap[:150])

        cursor = position + len(content)

    trailing = original_text[cursor:].strip()

    if trailing:
        unmatched_parts.append(trailing[:150])

    preservation_report.append({
        "smpc_id": smpc_id,
        "unmatched_parts": len(unmatched_parts),
        "examples": unmatched_parts[:3]
    })

preservation_df = pd.DataFrame(preservation_report)

print(
    "SmPCs with unmatched text:",
    (preservation_df["unmatched_parts"] > 0).sum()
)

display(
    preservation_df[
        preservation_df["unmatched_parts"] > 0
    ]
)

SmPCs with unmatched text: 0


,smpc_id,unmatched_parts,examples


#5. Save data: sections.csv

In [ ]:
from pathlib import Path

# Save extracted sections to Google Drive
output_dir = DATA_DIR / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "sections.csv"

sections_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("CSV saved successfully!")
print("Location:", output_path)
print("Total rows:", len(sections_df))
print("Unique drugs:", sections_df["drug"].nunique())
print("Unique SmPCs:", sections_df["smpc_id"].nunique())

CSV saved successfully!
Location: /content/drive/MyDrive/AI MACHINE PROJECT/data/processed/sections.csv
Total rows: 1680
Unique drugs: 30
Unique SmPCs: 60


#6. Validate that all the data in the csv = pdf

In [22]:
import re
import fitz
import pandas as pd

# Important sections for medication safety
CRITICAL_SECTIONS = ["4.2", "4.3", "4.4", "4.5", "4.6", "4.8"]

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()

# Load and normalize original PDF text
pdf_texts = {}

for pdf_path in pdf_files:
    with fitz.open(pdf_path) as doc:
        full_text = " ".join(
            page.get_text("text") for page in doc
        )
    pdf_texts[pdf_path.name] = normalize_text(full_text)

# Compare extracted sections against original PDF content
validation_results = []

for _, row in sections_df.iterrows():

    if str(row["section_number"]) not in CRITICAL_SECTIONS:
        continue

    source_pdf = row["source_pdf"]
    extracted_text = normalize_text(row["text"])
    original_text = pdf_texts[source_pdf]

    # Use the beginning, middle and end of each section
    sample_size = 250

    samples = [
        extracted_text[:sample_size],
        extracted_text[
            max(0, len(extracted_text)//2 - sample_size//2):
            len(extracted_text)//2 + sample_size//2
        ],
        extracted_text[-sample_size:]
    ]

    matches = [
        sample in original_text
        for sample in samples
    ]

    validation_results.append({
        "drug": row["drug"],
        "smpc_id": row["smpc_id"],
        "section": row["section_number"],
        "start_match": matches[0],
        "middle_match": matches[1],
        "end_match": matches[2],
        "all_match": all(matches)
    })

clinical_validation_df = pd.DataFrame(validation_results)

print("Critical sections checked:", len(clinical_validation_df))

print(
    "Sections with all samples matching:",
    clinical_validation_df["all_match"].sum()
)

print(
    "Sections requiring review:",
    (~clinical_validation_df["all_match"]).sum()
)

display(
    clinical_validation_df[
        ~clinical_validation_df["all_match"]
    ]
)

Critical sections checked: 360
Sections with all samples matching: 360
Sections requiring review: 0


,drug,smpc_id,section,start_match,middle_match,end_match,all_match


In [23]:
# Validate boundaries between SmPCs

import re
import pandas as pd

boundary_results = []

for _, smpc in smpcs_clean.iterrows():

    smpc_id = smpc["smpc_id"]
    text = smpc["text"]

    # Identify medicinal product name from section 1
    section_1 = sections_df[
        (sections_df["smpc_id"] == smpc_id) &
        (sections_df["section_number"] == "1")
    ]

    product_name = (
        section_1.iloc[0]["text"].splitlines()[1].strip()
        if len(section_1) > 0
        and len(section_1.iloc[0]["text"].splitlines()) > 1
        else "REVIEW"
    )

    # Count occurrences of section 1 heading
    section_1_count = len(re.findall(
        r"(?im)^[ \t]*1\.[ \t]*\n"
        r"[ \t]*NAME OF THE MEDICINAL PRODUCT",
        text
    ))

    # Count occurrences of section 10 heading
    section_10_count = len(re.findall(
        r"(?im)^[ \t]*10\.[ \t]*\n"
        r"[ \t]*DATE OF REVISION OF THE TEXT",
        text
    ))

    boundary_results.append({
        "drug": smpc["drug"],
        "smpc_id": smpc_id,
        "product_name": product_name,
        "section_1_count": section_1_count,
        "section_10_count": section_10_count,
        "valid_boundaries": (
            section_1_count == 1 and section_10_count == 1
        )
    })

boundary_df = pd.DataFrame(boundary_results)

print("SmPCs checked:", len(boundary_df))
print(
    "SmPCs with valid boundaries:",
    boundary_df["valid_boundaries"].sum()
)
print(
    "SmPCs requiring review:",
    (~boundary_df["valid_boundaries"]).sum()
)

print("\nSmPCs requiring review:")
display(boundary_df[~boundary_df["valid_boundaries"]])

print("\nMulti-SmPC medicines:")
display(
    boundary_df[
        boundary_df["drug"].isin([
            "Abasaglar", "Aprovel", "Brilique",
            "Eliquis", "Entresto", "Insuman",
            "Pradaxa", "Ranexa", "Xarelto"
        ])
    ][["drug", "smpc_id", "product_name"]]
)

SmPCs checked: 60
SmPCs with valid boundaries: 60
SmPCs requiring review: 0

SmPCs requiring review:


,drug,smpc_id,product_name,section_1_count,section_10_count,valid_boundaries



Multi-SmPC medicines:


,drug,smpc_id,product_name
0,Abasaglar,abasaglar_01,
1,Abasaglar,abasaglar_02,
3,Aprovel,aprovel_01,
4,Aprovel,aprovel_02,
5,Aprovel,aprovel_03,
6,Aprovel,aprovel_04,
7,Aprovel,aprovel_05,
8,Aprovel,aprovel_06,
9,Brilique,brilique_01,
10,Brilique,brilique_02,


#7. Extract product name

In [24]:
# Identify the exact medicinal product name for each SmPC

product_results = []

for _, smpc in smpcs_clean.iterrows():

    smpc_id = smpc["smpc_id"]

    section_1 = sections_df[
        (sections_df["smpc_id"] == smpc_id) &
        (sections_df["section_number"] == "1")
    ]

    text = section_1.iloc[0]["text"]

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    # Remove the section title if present
    lines = [
        line for line in lines
        if "NAME OF THE MEDICINAL PRODUCT" not in line.upper()
    ]

    product_name = lines[0] if lines else "REVIEW"

    product_results.append({
        "drug": smpc["drug"],
        "smpc_id": smpc_id,
        "product_name": product_name
    })

products_df = pd.DataFrame(product_results)

print("Total SmPCs:", len(products_df))
print("Missing product names:",
      (products_df["product_name"] == "REVIEW").sum())

display(products_df)

Total SmPCs: 60
Missing product names: 0


,drug,smpc_id,product_name
0,Abasaglar,abasaglar_01,ABASAGLAR 100 units/mL solution for injection in a cartridge
1,Abasaglar,abasaglar_02,ABASAGLAR 100 units/mL KwikPen solution for injection in a pre-filled pen
2,Actos,actos_01,Actos 15 mg tablets
3,Aprovel,aprovel_01,Aprovel 75 mg tablets.
4,Aprovel,aprovel_02,Aprovel 150 mg tablets.
5,Aprovel,aprovel_03,Aprovel 300 mg tablets.
6,Aprovel,aprovel_04,Aprovel 75 mg film-coated tablets.
7,Aprovel,aprovel_05,Aprovel 150 mg film-coated tablets.
8,Aprovel,aprovel_06,Aprovel 300 mg film-coated tablets.
9,Brilique,brilique_01,Brilique 60 mg film-coated tablets


We can see that there are two SmPCs with the same product name: Xarelto 15 mg film-coated tablets

# 8. Evaluate Xarelto 15 mg

In [26]:
# Compare the two Xarelto 15 mg SmPCs

a = smpcs_clean.loc[
    smpcs_clean["smpc_id"] == "xarelto_03", "text"
].iloc[0]

b = smpcs_clean.loc[
    smpcs_clean["smpc_id"] == "xarelto_05", "text"
].iloc[0]

from difflib import SequenceMatcher

similarity = SequenceMatcher(
    None, a, b, autojunk=False
).ratio()

print("Xarelto 03 characters:", len(a))
print("Xarelto 05 characters:", len(b))
print("Exactly identical:", a == b)
print("Similarity:", round(similarity * 100, 2), "%")

print("\nSection 4.1 — Therapeutic indications")

for smpc_id in ["xarelto_03", "xarelto_05"]:
    section = sections_df[
        (sections_df["smpc_id"] == smpc_id) &
        (sections_df["section_number"] == "4.1")
    ]

    print(f"\n{smpc_id}:")
    print(section.iloc[0]["text"][:1200])

Xarelto 03 characters: 100384
Xarelto 05 characters: 69943
Exactly identical: False
Similarity: 80.41 %

Section 4.1 — Therapeutic indications

xarelto_03:
Therapeutic indications 
 
Adults 
Prevention of stroke and systemic embolism in adult patients with non-valvular atrial fibrillation with 
one or more risk factors, such as congestive heart failure, hypertension, age ≥ 75 years, diabetes 
mellitus, prior stroke or transient ischaemic attack.  
 
Treatment of deep vein thrombosis (DVT) and pulmonary embolism (PE), and prevention of recurrent 
DVT and PE in adults. (See section 4.4 for haemodynamically unstable PE patients.) 
 
Paediatric population  
Treatment of venous thromboembolism (VTE) and prevention of VTE recurrence in children and 
adolescents aged less than 18 years and weighing from 30 kg to 50 kg after at least 5 days of initial 
parenteral anticoagulation treatment.

xarelto_05:
Therapeutic indications 
 
Treatment of deep vein thrombosis (DVT) and pulmonary embolism (P

The two are not identical

In [30]:

import fitz

# Find the original PDF for Xarelto
pdf_path = next(
    p for p in pdf_files
    if p.stem.lower() == "xarelto"
)

doc = fitz.open(pdf_path)

# Search for the exact section 1 text of xarelto_05
section_text = sections_df.loc[
    (sections_df["smpc_id"] == "xarelto_05") &
    (sections_df["section_number"] == "1"),
    "text"
].iloc[0]

print("ORIGINAL PDF:", pdf_path.name)
print("\nSECTION 1 OF XARELTO_05:")
print(section_text)

print("\nMATCHING PDF PAGE:")

for page_number, page in enumerate(doc):
    page_text = page.get_text()

    if section_text[:100].strip() in page_text:
        print("PDF page:", page_number + 1)
        print(page_text[:1500])
        break

doc.close()


ORIGINAL PDF: xarelto.pdf

SECTION 1 OF XARELTO_05:
NAME OF THE MEDICINAL PRODUCT 
 
Xarelto 15 mg film-coated tablets 
Xarelto 20 mg film-coated tablets

MATCHING PDF PAGE:
PDF page: 118
118 
Treatment Initiation Pack 
 
1. 
NAME OF THE MEDICINAL PRODUCT 
 
Xarelto 15 mg film-coated tablets 
Xarelto 20 mg film-coated tablets 
 
 
2. 
QUALITATIVE AND QUANTITATIVE COMPOSITION 
 
Each 15 mg film-coated tablet contains 15 mg rivaroxaban.  
Each 20 mg film-coated tablet contains 20 mg rivaroxaban.  
 
Excipient with known effect 
Each 15 mg film-coated tablet contains 24.13 mg lactose (as monohydrate), see section 4.4. 
Each 20 mg film-coated tablet contains 21.76 mg lactose (as monohydrate), see section 4.4. 
 
For the full list of excipients, see section 6.1. 
 
 
3. 
PHARMACEUTICAL FORM 
 
Film-coated tablet (tablet)  
 
15 mg film-coated tablet: red, round biconvex tablets (6 mm diameter, 9 mm radius of curvature) 
marked with the BAYER-cross on one side and “15” and a triangle on the 

In [31]:
# Correct the product name for Xarelto Treatment Initiation Pack

corrected_name = (
    "Xarelto 15 mg + 20 mg film-coated tablets "
    "(Treatment Initiation Pack)"
)

# Update products_df
products_df.loc[
    products_df["smpc_id"] == "xarelto_05",
    "product_name"
] = corrected_name

# Update sections_df
sections_df["product_name"] = sections_df["smpc_id"].map(
    products_df.set_index("smpc_id")["product_name"]
)

# Verify the correction
display(
    sections_df[
        sections_df["smpc_id"].isin(["xarelto_03", "xarelto_05"])
    ][["smpc_id", "product_name"]].drop_duplicates()
)

print("Total sections:", len(sections_df))
print("Missing product names:", sections_df["product_name"].isna().sum())


,smpc_id,product_name
1540,xarelto_03,Xarelto 15 mg film-coated tablets
1596,xarelto_05,Xarelto 15 mg + 20 mg film-coated tablets (Treatment Initiation Pack)


Total sections: 1680
Missing product names: 0


#9. Change in csv

In [32]:
from pathlib import Path
import pandas as pd

csv_path = Path(
    "/content/drive/MyDrive/AI MACHINE PROJECT/data/processed/sections.csv"
)

# Overwrite the original CSV
sections_df.to_csv(csv_path, index=False, encoding="utf-8")

# Read it again to verify
saved_df = pd.read_csv(csv_path)

print("CSV updated successfully!")
print("Total sections:", len(saved_df))
print("Total SmPCs:", saved_df["smpc_id"].nunique())

display(
    saved_df[
        saved_df["smpc_id"].isin(["xarelto_03", "xarelto_05"])
    ][["smpc_id", "product_name"]].drop_duplicates()
)


CSV updated successfully!
Total sections: 1680
Total SmPCs: 60


,smpc_id,product_name
1540,xarelto_03,Xarelto 15 mg film-coated tablets
1596,xarelto_05,Xarelto 15 mg + 20 mg film-coated tablets (Treatment Initiation Pack)


In [36]:

import csv

with open(csv_path, encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        if row["smpc_id"] == "xarelto_05":
            print("Section number stored in CSV:",
                  repr(row["section_number"]))
            break


Section number stored in CSV: '1'


# TABLE EXTRACTION

In [38]:

import fitz
import pandas as pd
import json

all_tables = []
errors = []

# Extract tables from all 30 PDFs
for pdf_path in pdf_files:

    drug = pdf_path.stem
    print(f"Processing: {drug}")

    try:
        with fitz.open(pdf_path) as doc:

            for page_num, page in enumerate(doc):

                detected = page.find_tables()

                for table_num, table in enumerate(detected.tables):

                    data = table.extract()

                    # Skip completely empty tables
                    if not data or not any(
                        cell is not None and str(cell).strip()
                        for row in data
                        for cell in row
                    ):
                        continue

                    all_tables.append({
                        "drug": drug,
                        "source_pdf": pdf_path.name,
                        "page": page_num + 1,
                        "table_number": table_num + 1,
                        "rows": len(data),
                        "columns": max(
                            (len(row) for row in data),
                            default=0
                        ),
                        "table_data": json.dumps(
                            data,
                            ensure_ascii=False
                        )
                    })

    except Exception as e:
        errors.append({
            "drug": drug,
            "error": str(e)
        })

# Create DataFrame
tables_df = pd.DataFrame(
    all_tables,
    columns=[
        "drug",
        "source_pdf",
        "page",
        "table_number",
        "rows",
        "columns",
        "table_data"
    ]
)

print("\n" + "=" * 50)
print("PDFs processed:", len(pdf_files))
print("Tables extracted:", len(tables_df))
print("PDFs with errors:", len(errors))

print("\nTables per drug:")
display(
    tables_df.groupby("drug")
    .size()
    .reset_index(name="tables")
    .sort_values("tables", ascending=False)
)

if errors:
    print("\nExtraction errors:")
    display(pd.DataFrame(errors))


Processing: abasaglar
Processing: actos
Processing: aprovel
Processing: brilique
Processing: corlentor
Processing: duoplavin
Processing: efient
Processing: eliquis
Processing: entresto
Processing: eucreas
Processing: forxiga
Processing: insuman
Processing: jardiance
Processing: kerendia
Processing: leqvio
Processing: lixiana
Processing: micardis
Processing: mounjaro
Processing: multaq
Processing: nilemdo
Processing: ozempic
Processing: plavix
Processing: pradaxa
Processing: praluent
Processing: ranexa
Processing: repatha
Processing: synjardy
Processing: verquvo
Processing: xarelto
Processing: xelevia

PDFs processed: 30
Tables extracted: 617
PDFs with errors: 0

Tables per drug:


,drug,tables
22,pradaxa,115
28,xarelto,84
7,eliquis,58
17,mounjaro,47
11,insuman,38
3,brilique,28
15,lixiana,25
20,ozempic,20
12,jardiance,19
26,synjardy,17
